# Notebook 01: Empirical Stylized Facts of Equity Excess Growth Rates

This notebook reproduces the empirical analysis presented in the paper.
We characterize the three canonical stylized facts of financial returns using SPY daily data (2014-2024):

1. **Heavy-tailed (leptokurtic) distributions** - excess kurtosis >> 0
2. **Negligible linear autocorrelation** in raw returns
3. **Persistent volatility clustering** - slow ACF decay in |returns|

### Outputs
- **Figure 1**: Four-panel stylized facts visualization (distribution, Q-Q plot, returns ACF, |returns| ACF)
- **Table 1**: Descriptive statistics and stylized-facts tests (IS and OoS)

## Setup

In [ ]:
include("../Include.jl");

## Constants

In [ ]:
# --- CONSTANTS ---
risk_free_rate = 0.0;       # risk-free rate (set to 0 to isolate volatility patterns)
Δt = 1/252;                 # daily time step in years
ticker = "SPY";             # primary validation asset
L = 252;                    # ACF max lag (1 trading year)

## Load and Clean Data

In [ ]:
# --- LOAD IN-SAMPLE DATA (2014-2024) ---
train_dataset = MyPortfolioDataSet() |> x -> x["dataset"];
maximum_number_trading_days = nrow(train_dataset["AAPL"]);

# Filter to tickers with complete history
dataset = Dict{String,DataFrame}();
for (t, data) ∈ train_dataset
    if nrow(data) == maximum_number_trading_days
        dataset[t] = data;
    end
end

list_of_all_tickers = keys(dataset) |> collect |> sort;
println("In-sample: $(length(list_of_all_tickers)) tickers, $(maximum_number_trading_days) trading days each")

In [ ]:
# --- LOAD OUT-OF-SAMPLE DATA (2025) ---
oos_dataset_raw = MyOutOfSamplePortfolioDataSet() |> x -> x["dataset"];
oos_max_days = nrow(first(values(oos_dataset_raw)));

oos_dataset = Dict{String,DataFrame}();
for (t, data) ∈ oos_dataset_raw
    if nrow(data) == oos_max_days
        oos_dataset[t] = data;
    end
end

println("Out-of-sample: $(length(oos_dataset)) tickers, $(oos_max_days) trading days each")

## Compute Excess Growth Rates

In [ ]:
# --- COMPUTE EXCESS LOG GROWTH RATES ---
# In-sample
R_is = log_growth_matrix(dataset, ticker; Δt=Δt, risk_free_rate=risk_free_rate);

# Out-of-sample
R_oos = log_growth_matrix(oos_dataset, ticker; Δt=Δt, risk_free_rate=risk_free_rate);

println("IS: T = $(length(R_is)) observations")
println("OoS: T = $(length(R_oos)) observations")

## Table 1: Descriptive Statistics and Stylized-Facts Tests

Reproduces Table 1 from the paper. Tests:
- **Jarque-Bera**: tests normality (rejects if heavy tails or skewness)
- **Ljung-Box on G_t**: tests autocorrelation in raw returns (should reject due to market microstructure, or not)
- **Ljung-Box on |G_t|**: tests volatility clustering (should reject strongly)

In [ ]:
# --- TABLE 1: DESCRIPTIVE STATISTICS ---
function compute_descriptive_stats(R::Vector{Float64}, label::String)
    n = length(R);
    μ = mean(R);
    σ = std(R);
    skew = sum(((R .- μ) ./ σ).^3) / n;
    kurt_excess = sum(((R .- μ) ./ σ).^4) / n - 3.0;
    
    # Jarque-Bera test
    jb_stat = (n / 6) * (skew^2 + (kurt_excess^2) / 4);
    
    # Ljung-Box test on raw returns (lag 20)
    acf_raw = autocor(R, 1:20);
    lb_raw = n * (n + 2) * sum(acf_raw.^2 ./ (n .- (1:20)));
    
    # Ljung-Box test on |returns| (lag 20)
    acf_abs = autocor(abs.(R), 1:20);
    lb_abs = n * (n + 2) * sum(acf_abs.^2 ./ (n .- (1:20)));
    
    println("\n--- $label (T = $n) ---")
    println("Mean (annualized, %):     $(round(μ, digits=2))")
    println("Std Dev (annualized, %):  $(round(σ, digits=2))")
    println("Skewness:                 $(round(skew, digits=3))")
    println("Excess Kurtosis:          $(round(kurt_excess, digits=3))")
    println("JB statistic:             $(round(jb_stat, digits=1)) (critical value ≈ 5.99 at α=0.05)")
    println("LB on Gₜ (lag 20):        $(round(lb_raw, digits=1)) (critical value ≈ 31.4 at α=0.05)")
    println("LB on |Gₜ| (lag 20):      $(round(lb_abs, digits=1)) (critical value ≈ 31.4 at α=0.05)")
    
    return (mean=μ, std=σ, skewness=skew, excess_kurtosis=kurt_excess)
end

stats_is = compute_descriptive_stats(R_is, "In-Sample (2014-2024)");
stats_oos = compute_descriptive_stats(R_oos, "Out-of-Sample (2025)");

## Figure 1: Empirical Stylized Facts of SPY

Four-panel figure showing:
- **(a)** Marginal distribution with Gaussian and Laplace fits
- **(b)** Normal Q-Q plot (heavy tails visible as departures)
- **(c)** ACF of raw returns (near zero at all lags)
- **(d)** ACF of |returns| (slow decay = volatility clustering)

In [ ]:
# --- FIGURE 1: STYLIZED FACTS ---

# Panel (a): Marginal distribution with Gaussian and Laplace fits
μ_gauss = mean(R_is);
σ_gauss = std(R_is);
d_gauss = Normal(μ_gauss, σ_gauss);

# Fit Laplace via MLE: μ = median, b = mean(|x - median|)
μ_lap = median(R_is);
b_lap = mean(abs.(R_is .- μ_lap));
d_laplace = Laplace(μ_lap, b_lap);

x_grid = range(-800, 800, length=1000);

p1 = histogram(R_is, normalize=true, bins=150, alpha=0.4, color=:gray, label="Observed",
    title="(a) Marginal Distribution", titlefontsize=10, xlabel="Excess Growth Rate", ylabel="Density");
plot!(p1, x_grid, pdf.(d_gauss, x_grid), lw=2, color=:blue, label="Gaussian", ls=:dash);
plot!(p1, x_grid, pdf.(d_laplace, x_grid), lw=2, color=:red, label="Laplace");
xlims!(p1, -800, 800);

# Panel (b): Normal Q-Q plot
sorted_R = sort(R_is);
n = length(sorted_R);
theoretical_quantiles = [quantile(d_gauss, (i - 0.5) / n) for i in 1:n];

p2 = scatter(theoretical_quantiles, sorted_R, ms=1, alpha=0.5, color=:steelblue, label="",
    title="(b) Normal Q-Q Plot", titlefontsize=10, xlabel="Theoretical Quantiles", ylabel="Sample Quantiles");
qq_lims = [-600, 600];
plot!(p2, qq_lims, qq_lims, lw=2, color=:red, ls=:dash, label="45° line");

# Panel (c): ACF of raw returns
τ = 1:(L-1);
ci = 2.576 / sqrt(n);  # 99% CI

acf_raw = autocor(R_is, τ);
p3 = plot(τ, acf_raw, linetype=:steppost, lw=2, color=:steelblue, label="ACF(Gₜ)",
    title="(c) Returns Autocorrelation", titlefontsize=10, xlabel="Lag (trading days)", ylabel="ACF");
plot!(p3, τ, ci .* ones(length(τ)), lw=1.5, color=:gray, ls=:dash, label="99% CI");
plot!(p3, τ, -ci .* ones(length(τ)), lw=1.5, color=:gray, ls=:dash, label="");

# Panel (d): ACF of absolute returns
acf_abs = autocor(abs.(R_is), τ);
p4 = plot(τ, acf_abs, linetype=:steppost, lw=2, color=:darkorange, label="ACF(|Gₜ|)",
    title="(d) Volatility Clustering", titlefontsize=10, xlabel="Lag (trading days)", ylabel="ACF");
plot!(p4, τ, ci .* ones(length(τ)), lw=1.5, color=:gray, ls=:dash, label="99% CI");
plot!(p4, τ, -ci .* ones(length(τ)), lw=1.5, color=:gray, ls=:dash, label="");

# Combine into 2x2 figure
fig1 = plot(p1, p2, p3, p4, layout=(2,2), size=(1000, 700),
    plot_title="Figure 1: Empirical Stylized Facts of SPY Daily Excess Growth Rates (2014-2024)",
    plot_titlefontsize=12);
display(fig1)

# Save
savefig(fig1, joinpath(_PATH_TO_FIGURES, "Fig-1-Stylized-Facts-SPY.svg"))
savefig(fig1, joinpath(_PATH_TO_FIGURES, "Fig-1-Stylized-Facts-SPY.pdf"))

## Out-of-Sample Stylized Facts Comparison

Verify that the same stylized facts hold in the 2025 holdout window.

In [ ]:
# --- OOS STYLIZED FACTS VERIFICATION ---
τ_oos = 1:min(L-1, length(R_oos)-1);
ci_oos = 2.576 / sqrt(length(R_oos));

acf_oos_raw = autocor(R_oos, τ_oos);
acf_oos_abs = autocor(abs.(R_oos), τ_oos);

p_oos1 = histogram(R_oos, normalize=true, bins=50, alpha=0.4, color=:gray, label="OoS Observed",
    title="OoS Distribution (2025)", titlefontsize=10, xlabel="Excess Growth Rate", ylabel="Density");

p_oos2 = plot(τ_oos, acf_oos_raw, linetype=:steppost, lw=2, color=:steelblue, label="ACF(Gₜ) OoS",
    title="OoS Returns ACF", titlefontsize=10, xlabel="Lag", ylabel="ACF");
plot!(p_oos2, τ_oos, ci_oos .* ones(length(τ_oos)), lw=1.5, color=:gray, ls=:dash, label="99% CI");
plot!(p_oos2, τ_oos, -ci_oos .* ones(length(τ_oos)), lw=1.5, color=:gray, ls=:dash, label="");

p_oos3 = plot(τ_oos, acf_oos_abs, linetype=:steppost, lw=2, color=:darkorange, label="ACF(|Gₜ|) OoS",
    title="OoS Volatility Clustering", titlefontsize=10, xlabel="Lag", ylabel="ACF");
plot!(p_oos3, τ_oos, ci_oos .* ones(length(τ_oos)), lw=1.5, color=:gray, ls=:dash, label="99% CI");
plot!(p_oos3, τ_oos, -ci_oos .* ones(length(τ_oos)), lw=1.5, color=:gray, ls=:dash, label="");

fig_oos = plot(p_oos1, p_oos2, p_oos3, layout=(1,3), size=(1200, 350),
    plot_title="Out-of-Sample Stylized Facts (2025)", plot_titlefontsize=12);
display(fig_oos)

## Disclaimer

This content is offered solely for research and educational purposes. It does not constitute financial advice, investment recommendations, or trading strategies.